In [1]:
# ================================================================
# CABITZA ET AL. COVID-19 MODEL REPRODUCTION
# MASTER VALIDATION COMPARISON
# LOGISTIC REGRESSION + RANDOM FOREST
# ================================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ================================================================
# 1. PATHS
# ================================================================

PROJECT_ROOT = Path(
    r"C:\Users\bl\Documents\ML-Projects\covid-laboratory-prediction"
)

RESULTS_DIR = PROJECT_ROOT / "results" / "reproduction"

OUTPUT_PATH = (
    RESULTS_DIR /
    "lr_rf_master_validation_comparison.csv"
)

print("=" * 80)
print("CABITZA ET AL. — LR/RF MASTER VALIDATION COMPARISON")
print("=" * 80)


# ================================================================
# 2. LOAD OUR EXISTING REPRODUCED RESULTS
# ================================================================

lr_holdout = pd.read_csv(
    RESULTS_DIR /
    "logistic_regression_holdout_results.csv"
)

rf_holdout = pd.read_csv(
    RESULTS_DIR /
    "random_forest_holdout_results.csv"
)

internal_external = pd.read_csv(
    RESULTS_DIR /
    "lr_rf_internal_external_validation_summary.csv"
)

external = pd.read_csv(
    RESULTS_DIR /
    "lr_rf_external_validation_summary.csv"
)

print("\nExisting result files loaded successfully.")


# ================================================================
# 3. HELPER: EXTRACT HOLDOUT RESULTS
#
# We already know the reproduced values from notebooks 02 and 03.
# These are entered explicitly here so that the master table remains
# transparent and independent of differences in CSV column layout.
# ================================================================

reproduced_holdout = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "validation_stage": "Development hold-out",
        "source": "Reproduced",
        "n": 325,
        "auc": 0.8770,
        "accuracy": 0.8000,
        "sensitivity": 0.8089,
        "specificity": 0.7917,
        "brier_score": 0.1417
    },
    {
        "model": "Random Forest",
        "validation_stage": "Development hold-out",
        "source": "Reproduced",
        "n": 325,
        "auc": 0.8809,
        "accuracy": 0.8062,
        "sensitivity": 0.7962,
        "specificity": 0.8155,
        "brier_score": 0.1411
    }
])


# ================================================================
# 4. PUBLISHED DEVELOPMENT/HOLD-OUT RESULTS
#
# Standard COVID-specific models reported by Cabitza et al.
# These values were previously compared in notebooks 02 and 03.
# ================================================================

published_holdout = pd.DataFrame([
    {
        "model": "Logistic Regression",
        "validation_stage": "Development hold-out",
        "source": "Published",
        "n": np.nan,
        "auc": 0.83,
        "accuracy": 0.83,
        "sensitivity": 0.85,
        "specificity": 0.82,
        "brier_score": 0.14
    },
    {
        "model": "Random Forest",
        "validation_stage": "Development hold-out",
        "source": "Published",
        "n": np.nan,
        "auc": 0.84,
        "accuracy": 0.84,
        "sensitivity": 0.84,
        "specificity": 0.84,
        "brier_score": 0.12
    }
])


# ================================================================
# 5. INTERNAL-EXTERNAL VALIDATION
#
# Extract mean performance across the 100 repeated IOG splits.
# ================================================================

def get_ie_metric(model, metric):

    row = internal_external[
        (internal_external["model"] == model) &
        (internal_external["metric"] == metric)
    ]

    if len(row) == 0:
        return np.nan

    return float(
        row.iloc[0]["mean"]
    )


ie_rows = []

for model in [
    "Logistic Regression",
    "Random Forest"
]:

    ie_rows.append({
        "model": model,
        "validation_stage":
            "Internal-external validation",
        "source": "Reproduced",
        "n": 58,
        "auc": get_ie_metric(
            model,
            "auc"
        ),
        "accuracy": get_ie_metric(
            model,
            "accuracy"
        ),
        "sensitivity": get_ie_metric(
            model,
            "sensitivity"
        ),
        "specificity": get_ie_metric(
            model,
            "specificity"
        ),
        "brier_score": get_ie_metric(
            model,
            "brier_score"
        )
    })

reproduced_internal_external = pd.DataFrame(
    ie_rows
)


# ================================================================
# 6. EXTERNAL VALIDATION
#
# Historical 2018 cohort:
# 54 cases, all negative.
#
# AUC and sensitivity cannot be estimated.
# Accuracy equals specificity because all cases are negative.
# ================================================================

external_rows = []

for model in [
    "Logistic Regression",
    "Random Forest"
]:

    row = external[
        external["model"] == model
    ].iloc[0]

    external_rows.append({
        "model": model,
        "validation_stage":
            "Historical external validation",
        "source": "Reproduced",
        "n": int(
            row["n_external"]
        ),
        "auc": np.nan,
        "accuracy": float(
            row["accuracy"]
        ),
        "sensitivity": np.nan,
        "specificity": float(
            row["specificity"]
        ),
        "brier_score": float(
            row["brier_score"]
        )
    })

reproduced_external = pd.DataFrame(
    external_rows
)


# ================================================================
# 7. BUILD MASTER TABLE
# ================================================================

master = pd.concat(
    [
        published_holdout,
        reproduced_holdout,
        reproduced_internal_external,
        reproduced_external
    ],
    ignore_index=True
)


# ================================================================
# 8. ADD NOTES
# ================================================================

def add_note(row):

    if (
        row["validation_stage"] ==
        "Development hold-out"
        and row["source"] == "Published"
    ):
        return (
            "Standard COVID-specific model performance "
            "reported by Cabitza et al."
        )

    if (
        row["validation_stage"] ==
        "Development hold-out"
        and row["source"] == "Reproduced"
    ):
        return (
            "80/20 stratified hold-out from the "
            "1,624-case OSR development cohort."
        )

    if (
        row["validation_stage"] ==
        "Internal-external validation"
    ):
        return (
            "Mean of 100 repeated stratified 50/50 IOG "
            "splits; public CSV contains 28 negative and "
            "30 positive IOG cases rather than the "
            "published 29/29 distribution."
        )

    if (
        row["validation_stage"] ==
        "Historical external validation"
    ):
        return (
            "54 historical 2018 cases, all COVID-negative; "
            "AUC and sensitivity are not estimable and "
            "accuracy equals specificity."
        )

    return ""


master["notes"] = master.apply(
    add_note,
    axis=1
)


# ================================================================
# 9. ORDER TABLE
# ================================================================

stage_order = {
    "Development hold-out": 1,
    "Internal-external validation": 2,
    "Historical external validation": 3
}

source_order = {
    "Published": 1,
    "Reproduced": 2
}

model_order = {
    "Logistic Regression": 1,
    "Random Forest": 2
}

master["_model_order"] = (
    master["model"].map(model_order)
)

master["_stage_order"] = (
    master["validation_stage"].map(stage_order)
)

master["_source_order"] = (
    master["source"].map(source_order)
)

master = master.sort_values(
    [
        "_model_order",
        "_stage_order",
        "_source_order"
    ]
).drop(
    columns=[
        "_model_order",
        "_stage_order",
        "_source_order"
    ]
).reset_index(
    drop=True
)


# ================================================================
# 10. DISPLAY MASTER TABLE
# ================================================================

display_columns = [
    "model",
    "validation_stage",
    "source",
    "n",
    "auc",
    "accuracy",
    "sensitivity",
    "specificity",
    "brier_score"
]

print("\n" + "=" * 80)
print("MASTER VALIDATION TABLE")
print("=" * 80)

print(
    master[
        display_columns
    ].round(4).to_string(
        index=False
    )
)


# ================================================================
# 11. DEVELOPMENT: PUBLISHED VS REPRODUCED DIFFERENCES
# ================================================================

print("\n" + "=" * 80)
print("PUBLISHED VS REPRODUCED DEVELOPMENT HOLD-OUT")
print("=" * 80)

comparison_rows = []

for model in [
    "Logistic Regression",
    "Random Forest"
]:

    published = master[
        (master["model"] == model) &
        (master["validation_stage"] ==
         "Development hold-out") &
        (master["source"] == "Published")
    ].iloc[0]

    reproduced = master[
        (master["model"] == model) &
        (master["validation_stage"] ==
         "Development hold-out") &
        (master["source"] == "Reproduced")
    ].iloc[0]

    for metric in [
        "auc",
        "accuracy",
        "sensitivity",
        "specificity",
        "brier_score"
    ]:

        comparison_rows.append({
            "model": model,
            "metric": metric,
            "published": published[metric],
            "reproduced": reproduced[metric],
            "difference_reproduced_minus_published":
                reproduced[metric] -
                published[metric]
        })


development_comparison = pd.DataFrame(
    comparison_rows
)

print(
    development_comparison.round(4).to_string(
        index=False
    )
)


# ================================================================
# 12. MODEL TRAJECTORY
# ================================================================

print("\n" + "=" * 80)
print("REPRODUCED VALIDATION TRAJECTORY")
print("=" * 80)

trajectory = master[
    master["source"] == "Reproduced"
][
    display_columns
].copy()

print(
    trajectory.round(4).to_string(
        index=False
    )
)


# ================================================================
# 13. SAVE MASTER TABLES
# ================================================================

master.to_csv(
    OUTPUT_PATH,
    index=False
)

development_comparison.to_csv(
    RESULTS_DIR /
    "lr_rf_published_vs_reproduced_comparison.csv",
    index=False
)

trajectory.to_csv(
    RESULTS_DIR /
    "lr_rf_validation_trajectory.csv",
    index=False
)


print("\n" + "=" * 80)
print("FILES SAVED")
print("=" * 80)

print(
    OUTPUT_PATH
)

print(
    RESULTS_DIR /
    "lr_rf_published_vs_reproduced_comparison.csv"
)

print(
    RESULTS_DIR /
    "lr_rf_validation_trajectory.csv"
)

print("\nMaster LR/RF comparison complete.")

CABITZA ET AL. — LR/RF MASTER VALIDATION COMPARISON

Existing result files loaded successfully.

MASTER VALIDATION TABLE
              model               validation_stage     source     n    auc  accuracy  sensitivity  specificity  brier_score
Logistic Regression           Development hold-out  Published   NaN 0.8300    0.8300       0.8500       0.8200       0.1400
Logistic Regression           Development hold-out Reproduced 325.0 0.8770    0.8000       0.8089       0.7917       0.1417
Logistic Regression   Internal-external validation Reproduced  58.0 0.8949    0.8107       0.7027       0.9264       0.1419
Logistic Regression Historical external validation Reproduced  54.0    NaN    0.9074          NaN       0.9074       0.0709
      Random Forest           Development hold-out  Published   NaN 0.8400    0.8400       0.8400       0.8400       0.1200
      Random Forest           Development hold-out Reproduced 325.0 0.8809    0.8062       0.7962       0.8155       0.1411
      Rando